# №2 СӨЖ — ЭКГ сигналдарын CNN арқылы жіктеу

Гиперпараметрлер: learning rate, Conv1D қабаттарының саны және фильтрлер саны. **Grid Search және Random Search**. Нәтижелер іске қосылған кезде есептеледі; алдын ала ойдан жазылмаған.

Дерек: [MIT-BIH Heartbeat Categorization](https://www.kaggle.com/datasets/shayanfazeli/heartbeat).

In [ ]:
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt, time
from sklearn.model_selection import train_test_split, ParameterSampler, ParameterGrid
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
SEED=42
tf.keras.utils.set_random_seed(SEED)

## 1. Деректерді Colab-қа жүктеу
`mitbih_train.csv` және `mitbih_test.csv` екеуін таңдаңыз.

In [ ]:
from pathlib import Path
try:
 from google.colab import files
 files.upload()
except ImportError: pass
train=pd.read_csv('mitbih_train.csv',header=None)
test=pd.read_csv('mitbih_test.csv',header=None)
print(train.shape,test.shape); display(train.head())

In [ ]:
X=train.iloc[:,:-1].to_numpy(dtype='float32'); y=train.iloc[:,-1].to_numpy(dtype='int64')
Xt=test.iloc[:,:-1].to_numpy(dtype='float32'); yt=test.iloc[:,-1].to_numpy(dtype='int64')
# Қысқа эксперимент: толық датасет үшін LIMIT=None
LIMIT=16000
if LIMIT and len(y)>LIMIT: X,_,y,_=train_test_split(X,y,train_size=LIMIT,stratify=y,random_state=SEED)
Xtr,Xv,ytr,yv=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED)
Xtr=Xtr[...,None]; Xv=Xv[...,None]; Xt=Xt[...,None]
print(Xtr.shape,Xv.shape,Xt.shape)

In [ ]:
fig, ax=plt.subplots(figsize=(10,3))
for k in np.unique(y): ax.plot(X[np.where(y==k)[0][0]],label=f'Класс {k}',alpha=.7)
ax.legend();ax.set(title='ЭКГ сигналдарының үлгілері',xlabel='Уақыт нүктесі',ylabel='Амплитуда');plt.show()

## 2. CNN архитектурасы
Conv1D уақыттық сигналдағы белгілерді анықтайды; MaxPooling өлшемді азайтады; Dense жіктейді.

In [ ]:
def build_model(learning_rate=.001,layers=2,filters=32):
 model=tf.keras.Sequential([tf.keras.layers.Input((187,1))])
 for i in range(layers):
  model.add(tf.keras.layers.Conv1D(filters,5,padding='same',activation='relu'))
  model.add(tf.keras.layers.MaxPooling1D(2))
 model.add(tf.keras.layers.GlobalAveragePooling1D())
 model.add(tf.keras.layers.Dense(32,activation='relu'))
 model.add(tf.keras.layers.Dense(5,activation='softmax'))
 model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
 return model
build_model().summary()

## 3. Grid Search — барлық комбинациялар

In [ ]:
grid={'learning_rate':[.001,.003],'layers':[1,2],'filters':[16,32]}
EPOCHS=4; BATCH_SIZE=128
def trial(parameters,method):
 tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(SEED)
 model=build_model(**parameters); t=time.perf_counter()
 model.fit(Xtr,ytr,validation_data=(Xv,yv),batch_size=BATCH_SIZE,epochs=EPOCHS,verbose=0)
 secs=time.perf_counter()-t
 pred=model.predict(Xv,verbose=0).argmax(axis=1)
 return {'method':method,**parameters,'accuracy':accuracy_score(yv,pred),'macro_f1':f1_score(yv,pred,average='macro',zero_division=0),'seconds':secs}
grid_rows=[]
for i,p in enumerate(ParameterGrid(grid),1):
 print('Grid',i,p)
 grid_rows.append(trial(p,'Grid Search'))
display(pd.DataFrame(grid_rows).sort_values('macro_f1',ascending=False))

## 4. Random Search — кездейсоқ комбинациялар
Екі іздеудің сынақ саны бірдей. Бірақ іздеу кеңістіктері әртүрлі, сондықтан бұл демонстрациялық салыстыру.

In [ ]:
random_space={'learning_rate':[.0003,.0005,.001,.002,.003],'layers':[1,2,3],'filters':[16,24,32,48,64]}
random_rows=[]
for i,p in enumerate(ParameterSampler(random_space,n_iter=len(grid_rows),random_state=SEED),1):
 print('Random',i,p)
 random_rows.append(trial(p,'Random Search'))
display(pd.DataFrame(random_rows).sort_values('macro_f1',ascending=False))

## 5. Нәтижелерді салыстыру

In [ ]:
results=pd.DataFrame(grid_rows+random_rows)
summary=results.sort_values('macro_f1',ascending=False).groupby('method',as_index=False).first()
summary['total_seconds']=summary['method'].map(results.groupby('method').seconds.sum())
display(summary)
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].bar(summary.method,summary.macro_f1);ax[0].set(title='Ең жақсы macro-F1',ylim=(0,1))
ax[1].bar(summary.method,summary.total_seconds);ax[1].set(title='Жалпы уақыт (сек)')
plt.tight_layout();plt.show()
results.to_csv('search_results.csv',index=False);summary.to_csv('summary_results.csv',index=False)

## 6. Финалдық CNN-ді тест жиынында бағалау
Тест жиынын гиперпараметр таңдау кезінде қолданбаймыз.

In [ ]:
best=results.sort_values('macro_f1',ascending=False).iloc[0]
params={'learning_rate':float(best.learning_rate),'layers':int(best.layers),'filters':int(best.filters)}
tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(SEED)
model=build_model(**params)
model.fit(np.concatenate([Xtr,Xv]),np.concatenate([ytr,yv]),batch_size=BATCH_SIZE,epochs=EPOCHS,verbose=1)
pred=model.predict(Xt,verbose=0).argmax(axis=1)
print('Test accuracy:',accuracy_score(yt,pred),'Test macro-F1:',f1_score(yt,pred,average='macro',zero_division=0))
print(classification_report(yt,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(yt,pred,cmap='Blues');plt.show()

## Қорытынды
Жоғарыдағы метрикаларға сүйеніп екі әдістің дәлдігі мен уақытын салыстырыңыз. Бұл бастапқы Kaggle авторының көшірмесі емес, деректерді пайдаланып жаңадан құрастырылған тәжірибе.